In [1]:
from glasspy.data import SciGlass

source = SciGlass(
    properties_cfg={"keep_all": True},
    compounds_cfg={"keep_all": True},
)

# Find all columns that contain "density" (case-insensitive)
density_cols = [c for c in source.data["property"].columns if "dens" in c.lower()]
print(density_cols)

['Density293K', 'Density1073K', 'Density1273K', 'Density1473K', 'Density1673K']


In [2]:
import sys
sys.path.append("..")

from src.data import load_glass_data, get_oxide_features

df = load_glass_data(target="Density293K", coverage_threshold=90.0)
print(df.shape)
print(df["Density293K"].describe())

(70548, 29)
count    70548.000000
mean         3.251497
std          1.147713
min          1.061684
25%          2.480000
50%          2.753000
75%          3.660000
max         11.532164
Name: Density293K, dtype: float64


In [3]:
from src.model import train_model, save_model

FEATURES = get_oxide_features()
X = df[FEATURES]
y = df["Density293K"]

param_distributions = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 20, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.3, 0.5],
}

model, results = train_model(X, y, param_distributions, n_iter=50)

print(f"Test RMSE: {results['test_rmse']:.4f}")
print(f"Test R²:   {results['test_r2']:.4f}")
print(f"Best params: {results['best_params']}")

Test RMSE: 0.2021
Test R²:   0.9688
Best params: {'n_estimators': 600, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 40}


In [4]:
save_model(
    model,
    path="../outputs/models/density293k_rf_tuned.joblib",
    metadata={
        "target": "Density293K",
        "features": FEATURES,
        "test_rmse": results["test_rmse"],
        "test_r2": results["test_r2"],
        "best_params": results["best_params"],
    }
)

Saved to ..\outputs\models\density293k_rf_tuned.joblib
